# AI-Powered Personal Task Manager

## GenAI Assignment Notebook — single-file, offline-first implementation

This notebook is **self-contained**: all the code it needs is written in its own cells, and it
runs top to bottom in a clean directory. It uses only the Python standard library, needs **no
API key and no network access**, and stores everything in a temporary SQLite database that is
removed again at the end.

A real **Google Gemini** integration is included as an *optional* section. It activates only
when you provide `GOOGLE_API_KEY` in the environment; the saved outputs contain no key and no
live API call.

## 1. Problem understanding and proposed approach

**Problem.** Build an assistant that accepts everyday natural-language instructions
("Add a task to call mom tomorrow morning", "Mark all tomorrow's tasks as done") and turns them
into reliable, validated task operations — without ever inventing information.

**Key challenges and how the design answers them**

| Challenge | Design decision |
|---|---|
| Times are expressed informally: `3 PM`, `3:30 PM`, `15:00`, `tomorrow morning` | Resolve relative dates against today; store exact times as 24-hour `HH:MM`; store approximate periods (`morning` / `afternoon` / `evening` / `night`) **separately** — an approximate period is never converted into an exact time |
| "at 3" is ambiguous (AM or PM?) | Ask a clarification question instead of guessing |
| Titles must stay clean: "Train at 3 PM on Saturday" → title `Train` | Date/time words are stripped from the title; the schedule lives in structured fields |
| A model could claim things it did not do | The model may only request one of **five predefined tools**; Python validates every argument, executes the operation, and builds the reply from the stored result |
| Bulk actions are risky | Bulk completion only *prepares* candidates; the tasks are completed only after an explicit "yes" |
| Users may ask for deletion | Deletion is **not implemented**; such requests are refused explicitly and nothing changes |
| A transient API failure must not look like success | Failed model calls execute no tool and never report task success; transient HTTP 429/503 errors get bounded retries with exponential backoff |

**Proposed approach.** A small layered pipeline:

```
natural language ──► LLM (mock or Gemini) ──► one of five tool calls
                                                     │
                             Python validates every argument
                                                     │
                       service layer (SQLite storage, confirmation state)
                                                     │
                          reply built from the actually stored result
```

The deterministic **offline mock** plays the role of the LLM so every demonstration is
reproducible; the optional Gemini client uses the same tool contract.

## 2. Technology stack and justification

| Choice | Why it fits this problem |
|---|---|
| **Python 3.10+** (developed on 3.12) | Clear, beginner-friendly, ideal for GenAI/function-calling examples; `zoneinfo` gives correct timezone handling |
| **SQLite** (standard library) | Zero-setup, durable, transactional storage — perfect for a single-user task manager and for a self-contained notebook |
| **Standard library only (notebook)** | Guarantees the notebook runs offline in a clean environment with nothing to install |
| **Function calling via tool schemas** | The LLM chooses *what* to do; Python stays in control of *how*, and validates every argument before execution |
| **Deterministic mock LLM** | Makes every demo and self-test reproducible without credentials or network; uses the same response shape as a real provider |
| **Google Gemini (optional)** | Gemini is the GitHub project's only supported real provider; here it is optional and its key is read from the environment only |

The same design is also implemented as a modular GitHub project (`src/`) with a 312-test pytest
suite, a CLI, and the same five tools. This notebook is an independent, condensed implementation
of that behavior — the two are consistent but do not depend on each other.

## 3. Design overview

**Data model** (one `tasks` table):

| Column | Meaning |
|---|---|
| `id` | Auto-incrementing primary key |
| `title` | Clean task title, without date/time words |
| `due_date` | Required ISO date `YYYY-MM-DD` |
| `due_time` | Optional exact time, normalized to 24-hour `HH:MM`; `NULL` when unknown |
| `due_period` | Optional approximate period: `morning`, `afternoon`, `evening`, `night`; `NULL` when unknown |
| `status` | `pending` or `completed` |
| `created_at` | Creation timestamp, used for stable ordering |

**Guarantees kept consistent with the GitHub project**

1. `due_time` and `due_period` are mutually exclusive, and a period is never turned into a time.
2. Ordering is chronological: due date, then untimed tasks before timed ones, then earlier
   times, then creation order.
3. Ambiguous task references ask the user which task is meant; ambiguous times ask AM or PM.
4. Bulk completion requires an explicit confirmation; "no" (or a new command) abandons it with
   no changes.
5. Deleting/clearing tasks is refused deterministically — before any model call — and no tool
   can delete records.

## 4. Python implementation — step by step

Each step below is a self-contained cell. Together they form a small but complete version of
the task manager.

### Step 1 — Setup and temporary storage

The notebook uses the standard library only, keeps the same `TIMEZONE` convention as the
project (default `UTC`), and creates its SQLite database inside a fresh temporary directory so
a real `tasks.db` is never touched.

In [1]:
# Standard-library imports only — nothing to install.
import json
import os
import re
import shutil
import sqlite3
import tempfile
import time
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from datetime import timezone as dt_timezone
from pathlib import Path
from typing import Optional
from zoneinfo import ZoneInfo

# The project requires an explicitly configured TIMEZONE and fails fast when
# it is invalid. The notebook keeps the same default (UTC) and safely falls
# back to UTC so it always runs top-to-bottom on a bare machine.
TIMEZONE_NAME = os.getenv("TIMEZONE") or "UTC"
try:
    LOCAL_TZ = ZoneInfo(TIMEZONE_NAME)
except Exception:
    LOCAL_TZ = dt_timezone.utc
    TIMEZONE_NAME = "UTC"

TODAY = datetime.now(LOCAL_TZ).date()
TOMORROW = TODAY + timedelta(days=1)

# Notebook-controlled temporary storage: the real ./tasks.db is never opened.
WORKDIR = Path(tempfile.mkdtemp(prefix="task_manager_notebook_"))
DB_PATH = WORKDIR / "notebook_tasks.db"

print("Runs fully offline: no API key, no network, standard library only.")
print(f"Timezone: {TIMEZONE_NAME} | Today: {TODAY.isoformat()} | Tomorrow: {TOMORROW.isoformat()}")
print(f"Demo database: {DB_PATH.name} (in a temporary directory)")

Runs fully offline: no API key, no network, standard library only.
Timezone: UTC | Today: 2026-10-19 | Tomorrow: 2026-10-20
Demo database: notebook_tasks.db (in a temporary directory)


### Step 2 — Date and time handling

Storage conventions: an exact time is normalized to 24-hour `HH:MM`, an approximate period is
stored as one of four words, and the two never mix.

In [2]:
TIME_PERIODS = ("morning", "afternoon", "evening", "night")

_PERIOD_ALIASES = {
    "morning": "morning", "am": "morning",
    "afternoon": "afternoon", "pm": "afternoon",
    "evening": "evening", "tonight": "evening",
    "night": "night",
}

_TIME_12H_RE = re.compile(r"^(\d{1,2})(?::(\d{2}))?\s*([ap])\.?m\.?$")
_TIME_24H_RE = re.compile(r"^(\d{1,2}):(\d{2})$")


def normalize_time_value(value):
    """Normalize '3 PM', '3:30 PM', '15:00' to 'HH:MM'; None for empty input."""
    if value is None:
        return None
    text = str(value).strip().lower()
    if not text:
        return None

    match = _TIME_12H_RE.match(text)
    if match:
        hour, minute, meridiem = int(match.group(1)), int(match.group(2) or 0), match.group(3)
        if not 1 <= hour <= 12:
            raise ValueError(f"Invalid time {value!r}: 12-hour clock hours must be 1-12.")
        if minute > 59:
            raise ValueError(f"Invalid time {value!r}: minutes must be 00-59.")
        if meridiem == "p" and hour != 12:
            hour += 12
        elif meridiem == "a" and hour == 12:
            hour = 0
        return f"{hour:02d}:{minute:02d}"

    match = _TIME_24H_RE.match(text)
    if match:
        hour, minute = int(match.group(1)), int(match.group(2))
        if hour > 23 or minute > 59:
            raise ValueError(f"Invalid time {value!r}: use 00:00-23:59.")
        return f"{hour:02d}:{minute:02d}"

    if text in {"noon", "midday"}:
        return "12:00"
    if text == "midnight":
        return "00:00"

    raise ValueError(
        f"Invalid time {value!r}. Use a 12-hour value such as '3 PM' or a "
        "24-hour value such as '15:00'."
    )


def normalize_period_value(value):
    """Normalize an approximate period to morning/afternoon/evening/night."""
    if value is None:
        return None
    text = str(value).strip().lower().rstrip(".")
    if not text:
        return None
    period = _PERIOD_ALIASES.get(text)
    if period is None:
        raise ValueError(
            f"Invalid period {value!r}. Expected one of: {', '.join(TIME_PERIODS)}."
        )
    return period


def format_time_12h(value):
    """Display a stored 'HH:MM' value as 'h:MM AM/PM'."""
    normalized = normalize_time_value(value)
    if normalized is None:
        return ""
    hour, minute = (int(part) for part in normalized.split(":"))
    meridiem = "AM" if hour < 12 else "PM"
    return f"{hour % 12 or 12}:{minute:02d} {meridiem}"


def format_due(due_date, due_time=None, due_period=None):
    """Human-readable due date, e.g. '2026-10-20 at 3:00 PM' or '2026-10-20 (evening)'."""
    base = due_date.isoformat() if isinstance(due_date, date) else str(due_date)
    if due_time:
        return f"{base} at {format_time_12h(str(due_time))}"
    if due_period:
        return f"{base} ({due_period})"
    return base


# Preview: exact times are stored in 24-hour form but displayed in 12-hour form.
for raw in ("3 PM", "3:30 PM", "15:00", "noon", "midnight"):
    stored = normalize_time_value(raw)
    print(f"{raw!r:10} -> stored {stored}  |  shown as {format_time_12h(stored)}")

'3 PM'     -> stored 15:00  |  shown as 3:00 PM
'3:30 PM'  -> stored 15:30  |  shown as 3:30 PM
'15:00'    -> stored 15:00  |  shown as 3:00 PM
'noon'     -> stored 12:00  |  shown as 12:00 PM
'midnight' -> stored 00:00  |  shown as 12:00 AM


### Step 3 — Storage layer (SQLite)

The schema matches the project, including the `CHECK` constraint that keeps periods within the
allowed set. Queries apply the same filter semantics: `due_date` alone is an exact match,
`due_date` + `end_date` is an inclusive range, and `end_date` alone means "on or before".

In [3]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    due_date TEXT NOT NULL,
    due_time TEXT,
    due_period TEXT CHECK (due_period IS NULL OR due_period IN ('morning', 'afternoon', 'evening', 'night')),
    status TEXT NOT NULL DEFAULT 'pending' CHECK (status IN ('pending', 'completed')),
    created_at TEXT NOT NULL
);
CREATE INDEX IF NOT EXISTS idx_tasks_status ON tasks(status);
CREATE INDEX IF NOT EXISTS idx_tasks_due_date ON tasks(due_date);
"""


@dataclass(frozen=True)
class Task:
    id: int
    title: str
    due_date: date
    due_time: Optional[str]
    due_period: Optional[str]
    status: str
    created_at: datetime

    @property
    def due_display(self):
        return format_due(self.due_date, self.due_time, self.due_period)


def get_connection(db_path=DB_PATH):
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    return conn


def initialize_database(db_path=DB_PATH):
    with get_connection(db_path) as conn:
        conn.executescript(SCHEMA_SQL)


def _row_to_task(row):
    return Task(
        id=row["id"],
        title=row["title"],
        due_date=date.fromisoformat(row["due_date"]),
        due_time=row["due_time"],
        due_period=row["due_period"],
        status=row["status"],
        created_at=datetime.fromisoformat(row["created_at"]),
    )


class TaskRepository:
    """Data access layer: create, filtered queries, title matching, completion."""

    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path
        initialize_database(db_path)

    def create_task(self, title, due_date, due_time=None, due_period=None):
        if not title or not title.strip():
            raise ValueError("Task title cannot be empty")
        now = datetime.now().isoformat()
        with get_connection(self.db_path) as conn:
            cursor = conn.execute(
                "INSERT INTO tasks (title, due_date, due_time, due_period, status, created_at) "
                "VALUES (?, ?, ?, ?, 'pending', ?)",
                (title.strip(), due_date.isoformat(), due_time, due_period, now),
            )
            conn.commit()
            return Task(
                id=cursor.lastrowid, title=title.strip(), due_date=due_date,
                due_time=due_time, due_period=due_period,
                status="pending", created_at=datetime.fromisoformat(now),
            )

    def get_pending_tasks_filtered(self, due_date=None, end_date=None,
                                   due_time=None, due_period=None, limit=None):
        """Pending tasks with the same filter and ordering rules as the project."""
        conditions, params = ["status = 'pending'"], []
        if due_date is not None and end_date is not None:
            conditions += ["due_date >= ?", "due_date <= ?"]
            params += [due_date.isoformat(), end_date.isoformat()]
        elif due_date is not None:
            conditions.append("due_date = ?")
            params.append(due_date.isoformat())
        elif end_date is not None:
            conditions.append("due_date <= ?")
            params.append(end_date.isoformat())
        if due_time is not None:
            conditions.append("due_time = ?")
            params.append(due_time)
        if due_period is not None:
            conditions.append("due_period = ?")
            params.append(due_period)

        limit_clause = ""
        if limit is not None:
            limit_clause = "LIMIT ?"
            params.append(int(limit))

        sql = f"""
            SELECT * FROM tasks
            WHERE {' AND '.join(conditions)}
            ORDER BY due_date ASC,
                     CASE WHEN due_time IS NULL THEN 0 ELSE 1 END ASC,
                     due_time ASC, created_at ASC, id ASC
            {limit_clause}
        """
        with get_connection(self.db_path) as conn:
            return tuple(_row_to_task(row) for row in conn.execute(sql, params).fetchall())

    def find_tasks_by_title(self, query):
        """Exact match first, then substring match, then all significant words."""
        if not query or not query.strip():
            return ()
        query = query.strip()
        stop_words = {"the", "a", "an", "my", "task", "to", "for", "of", "in",
                      "on", "at", "by", "with", "and", "or"}
        with get_connection(self.db_path) as conn:
            rows = conn.execute(
                "SELECT * FROM tasks WHERE LOWER(title) = LOWER(?) ORDER BY created_at DESC",
                (query,),
            ).fetchall()
            if rows:
                return tuple(_row_to_task(row) for row in rows)

            rows = conn.execute(
                "SELECT * FROM tasks WHERE LOWER(title) LIKE LOWER(?) ORDER BY created_at DESC",
                (f"%{query}%",),
            ).fetchall()
            if rows:
                return tuple(_row_to_task(row) for row in rows)

            words = [w for w in query.lower().split() if w not in stop_words and len(w) > 2]
            if words:
                conditions = " AND ".join(["LOWER(title) LIKE ?"] * len(words))
                rows = conn.execute(
                    f"SELECT * FROM tasks WHERE {conditions} GROUP BY id ORDER BY created_at DESC",
                    [f"%{word}%" for word in words],
                ).fetchall()
                if rows:
                    return tuple(_row_to_task(row) for row in rows)
        return ()

    def complete_task(self, task_id):
        with get_connection(self.db_path) as conn:
            row = conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
            if row is None:
                raise ValueError(f"Task with ID {task_id} not found")
            if row["status"] == "completed":
                raise ValueError(f"Task {task_id} is already completed")
            conn.execute("UPDATE tasks SET status = 'completed' WHERE id = ?", (task_id,))
            conn.commit()
            return _row_to_task(
                conn.execute("SELECT * FROM tasks WHERE id = ?", (task_id,)).fetchone()
            )

    def complete_tasks_by_ids(self, task_ids):
        """Complete the pending tasks among the given IDs in one transaction."""
        ids = [int(i) for i in task_ids]
        if not ids:
            raise ValueError("At least one task ID is required")
        placeholders = ", ".join("?" for _ in ids)
        with get_connection(self.db_path) as conn:
            pending = [row["id"] for row in conn.execute(
                f"SELECT id FROM tasks WHERE id IN ({placeholders}) AND status = 'pending'",
                ids,
            ).fetchall()]
            if not pending:
                return ()
            pending_placeholders = ", ".join("?" for _ in pending)
            conn.execute(
                f"UPDATE tasks SET status = 'completed' WHERE id IN ({pending_placeholders})",
                pending,
            )
            conn.commit()
            return tuple(_row_to_task(row) for row in conn.execute(
                f"SELECT * FROM tasks WHERE id IN ({pending_placeholders}) ORDER BY due_date, id",
                pending,
            ).fetchall())

    def count_tasks(self, status=None):
        with get_connection(self.db_path) as conn:
            if status:
                row = conn.execute(
                    "SELECT COUNT(*) AS n FROM tasks WHERE status = ?", (status,)
                ).fetchone()
            else:
                row = conn.execute("SELECT COUNT(*) AS n FROM tasks").fetchone()
            return row["n"]


DB = TaskRepository()
print(f"Database ready: {DB_PATH.name} (temporary directory)")
print("Schema columns: id, title, due_date, due_time, due_period, status, created_at")

Database ready: notebook_tasks.db (temporary directory)
Schema columns: id, title, due_date, due_time, due_period, status, created_at


### Step 4 — Service layer (validated operations)

The service is independent of any LLM. It validates inputs, calls the repository, and returns a
small structured result. Its messages match the GitHub project so both deliverables behave
consistently.

In [4]:
@dataclass(frozen=True)
class OperationResult:
    success: bool
    status: str
    message: str
    task: Optional[Task] = None
    tasks: tuple = ()
    matches: tuple = ()


class TaskService:
    def __init__(self, repository=DB):
        self.repo = repository

    # ---- create ------------------------------------------------------------
    def create_task(self, title, due_date, due_time=None, due_period=None):
        if not title or not title.strip():
            return OperationResult(False, "error", "Task title cannot be empty")
        if not isinstance(due_date, date):
            return OperationResult(False, "error", "Due date must be a valid date")
        try:
            normalized_time = normalize_time_value(due_time)
            normalized_period = normalize_period_value(due_period)
        except ValueError as exc:
            return OperationResult(False, "error", str(exc))
        if normalized_time and normalized_period:
            return OperationResult(
                False, "error",
                "Specify either an exact time or an approximate period, not both",
            )
        task = self.repo.create_task(
            title.strip(), due_date, due_time=normalized_time, due_period=normalized_period
        )
        return OperationResult(
            True, "created",
            f"Task created: {task.title} (due: {task.due_display}, status: {task.status})",
            task=task,
        )

    # ---- list --------------------------------------------------------------
    def list_pending_tasks(self, due_date=None, end_date=None,
                           due_time=None, due_period=None, limit=None):
        try:
            normalized_time = normalize_time_value(due_time)
            normalized_period = normalize_period_value(due_period)
        except ValueError as exc:
            return OperationResult(False, "error", str(exc))
        tasks = self.repo.get_pending_tasks_filtered(
            due_date=due_date, end_date=end_date,
            due_time=normalized_time, due_period=normalized_period, limit=limit,
        )
        if not tasks:
            return OperationResult(True, "listed", "No pending tasks found.")
        lines = [f"Pending tasks ({len(tasks)}):"]
        lines += [
            f"  [{t.id}] {t.title} — due: {t.due_display} — status: {t.status}"
            for t in tasks
        ]
        return OperationResult(True, "listed", "\n".join(lines), tasks=tasks)

    # ---- matching helper -----------------------------------------------------
    def _match(self, query):
        return self.repo.find_tasks_by_title(query.strip())

    # ---- single completion / details -------------------------------------------
    def complete_task(self, task_query):
        if not task_query or not task_query.strip():
            return OperationResult(False, "error", "Task query cannot be empty")
        matches = self._match(task_query)
        if not matches:
            return OperationResult(False, "not_found", f"Task not found: {task_query}")
        if len(matches) > 1:
            titles = ", ".join(f'"{t.title}"' for t in matches)
            return OperationResult(
                False, "ambiguous",
                f"Multiple tasks match: {titles}. Please specify.", matches=matches,
            )
        task = matches[0]
        if task.status == "completed":
            return OperationResult(
                False, "already_completed",
                f"Task already completed: {task.title}", task=task,
            )
        completed = self.repo.complete_task(task.id)
        return OperationResult(
            True, "completed", f"Task marked completed: {completed.title}", task=completed,
        )

    def get_task_details(self, task_query):
        if not task_query or not task_query.strip():
            return OperationResult(False, "error", "Task query cannot be empty")
        matches = self._match(task_query)
        if not matches:
            return OperationResult(False, "not_found", f"Task not found: {task_query}")
        if len(matches) > 1:
            titles = ", ".join(f'"{t.title}"' for t in matches)
            return OperationResult(
                False, "ambiguous",
                f"Multiple tasks match: {titles}. Please specify.", matches=matches,
            )
        task = matches[0]
        return OperationResult(
            True, "retrieved",
            f"Task details: {task.title} (id: {task.id}, due: {task.due_display}, "
            f"status: {task.status})",
            task=task,
        )

    # ---- confirmed bulk completion ----------------------------------------------
    def prepare_bulk_completion(self, due_date, end_date=None):
        label = due_date.isoformat()
        if end_date is not None:
            label = f"{due_date.isoformat()} to {end_date.isoformat()}"
        tasks = self.repo.get_pending_tasks_filtered(due_date=due_date, end_date=end_date)
        if not tasks:
            return OperationResult(False, "not_found", f"No pending tasks found for {label}.")
        lines = [f"I found {len(tasks)} pending task(s) for {label}:"]
        lines += [
            f"  {i}. [id {t.id}] {t.title} — due: {t.due_display}"
            for i, t in enumerate(tasks, 1)
        ]
        lines.append(f"Should I mark all {len(tasks)} as completed? (yes/no)")
        return OperationResult(True, "confirmation_required", "\n".join(lines), tasks=tasks)

    def complete_tasks_by_ids(self, task_ids):
        if not task_ids:
            return OperationResult(False, "error", "No task IDs were confirmed")
        completed = self.repo.complete_tasks_by_ids(task_ids)
        if not completed:
            return OperationResult(
                False, "error",
                "No pending tasks were completed; they may already be done.",
            )
        titles = ", ".join(t.title for t in completed)
        return OperationResult(
            True, "completed", f"Completed {len(completed)} task(s): {titles}.",
            tasks=completed,
        )


SERVICE = TaskService()
print("TaskService ready: create, list, details, single completion, confirmed bulk completion")

TaskService ready: create, list, details, single completion, confirmed bulk completion


### Step 5 — Tool layer (the model's only possible actions)

The LLM never touches SQLite. It can only request one of five predefined tools, and Python
validates each argument before execution. Model-invented tool names are rejected by the
allowlist. This is also where destructive requests are refused — before any model call.

In [5]:
TOOLS = [
    {"type": "function", "function": {
        "name": "create_task",
        "description": ("Create a task. Include due_time or due_period only when the user "
                        "gives one; never invent a time."),
        "parameters": {"type": "object", "properties": {
            "title": {"type": "string",
                      "description": "Task title, without date or time words"},
            "due_date": {"type": "string", "description": "ISO date YYYY-MM-DD"},
            "due_time": {"type": "string", "description": "Exact 24-hour time HH:MM"},
            "due_period": {"type": "string", "enum": list(TIME_PERIODS)},
        }, "required": ["title", "due_date"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "list_pending_tasks",
        "description": ("List pending tasks chronologically; optional date/time filters and "
                        "limit (limit=1 for the next task)."),
        "parameters": {"type": "object", "properties": {
            "due_date": {"type": "string", "description": "ISO date YYYY-MM-DD"},
            "end_date": {"type": "string", "description": "Inclusive end date for a range"},
            "due_time": {"type": "string", "description": "Exact 24-hour time HH:MM"},
            "due_period": {"type": "string", "enum": list(TIME_PERIODS)},
            "limit": {"type": "integer", "minimum": 1},
        }, "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "complete_task",
        "description": "Mark ONE task completed by title query.",
        "parameters": {"type": "object", "properties": {
            "task_query": {"type": "string"},
        }, "required": ["task_query"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "get_task_details",
        "description": "Get one task's due date/time or period and status by title query.",
        "parameters": {"type": "object", "properties": {
            "task_query": {"type": "string"},
        }, "required": ["task_query"], "additionalProperties": False}}},
    {"type": "function", "function": {
        "name": "complete_tasks_by_date",
        "description": ("Find pending tasks for a concrete date or range and ask the user to "
                        "confirm. Never completes anything by itself."),
        "parameters": {"type": "object", "properties": {
            "due_date": {"type": "string", "description": "ISO date YYYY-MM-DD"},
            "end_date": {"type": "string", "description": "Optional inclusive range end"},
        }, "required": ["due_date"], "additionalProperties": False}}},
]

TOOL_HANDLER_MAP = {
    "create_task": "create_task",
    "list_pending_tasks": "list_pending_tasks",
    "complete_task": "complete_task",
    "get_task_details": "get_task_details",
    "complete_tasks_by_date": "prepare_bulk_completion",
}


def validate_tool_arguments(tool_name, arguments):
    """Return (is_valid, error_message). Rejects unknown tools and bad arguments."""
    def valid_date(value):
        try:
            date.fromisoformat(value)
            return True
        except (TypeError, ValueError):
            return False

    if tool_name == "create_task":
        if not arguments.get("title") or not str(arguments["title"]).strip():
            return False, "Title cannot be empty"
        if not valid_date(arguments.get("due_date")):
            return False, f"Invalid due_date format: {arguments.get('due_date')}. Must be YYYY-MM-DD"
        has_time = bool(str(arguments.get("due_time") or "").strip())
        has_period = bool(str(arguments.get("due_period") or "").strip())
        if has_time and has_period:
            return False, "Provide either due_time or due_period, not both"
        try:
            normalize_time_value(arguments.get("due_time"))
            normalize_period_value(arguments.get("due_period"))
        except ValueError as exc:
            return False, str(exc)
        return True, ""

    if tool_name == "list_pending_tasks":
        for name in ("due_date", "end_date"):
            value = arguments.get(name)
            if value not in (None, "") and not valid_date(value):
                return False, f"Invalid {name} format: {value}. Must be YYYY-MM-DD"
        if (arguments.get("due_date") and arguments.get("end_date")
                and arguments["end_date"] < arguments["due_date"]):
            return False, "end_date cannot be earlier than due_date"
        limit = arguments.get("limit")
        if limit is not None and (isinstance(limit, bool)
                                  or not isinstance(limit, int) or limit < 1):
            return False, "limit must be a positive integer"
        return True, ""

    if tool_name in ("complete_task", "get_task_details"):
        if not arguments.get("task_query") or not str(arguments["task_query"]).strip():
            return False, "Task query cannot be empty"
        return True, ""

    if tool_name == "complete_tasks_by_date":
        if not valid_date(arguments.get("due_date")):
            return False, f"Invalid due_date format: {arguments.get('due_date')}. Must be YYYY-MM-DD"
        end_date = arguments.get("end_date")
        if end_date not in (None, "") and not valid_date(end_date):
            return False, f"Invalid end_date format: {end_date}. Must be YYYY-MM-DD"
        return True, ""

    return False, f"Unknown tool: {tool_name}"


def _tool_result(tool_call, tool_name, payload):
    return {"tool_call_id": tool_call.get("id", ""), "role": "tool",
            "name": tool_name, "content": json.dumps(payload)}


def execute_tool_call(tool_call, service):
    """Validate and execute one tool call; return a JSON tool result."""
    function = tool_call.get("function", {})
    tool_name = function.get("name", "")
    try:
        arguments = json.loads(function.get("arguments", "{}"))
    except json.JSONDecodeError as exc:
        return _tool_result(tool_call, tool_name,
                            {"success": False, "error": f"Invalid JSON: {exc}"})

    is_valid, error = validate_tool_arguments(tool_name, arguments)
    if not is_valid:
        return _tool_result(tool_call, tool_name, {"success": False, "error": error})

    handler_name = TOOL_HANDLER_MAP.get(tool_name)
    if not handler_name:
        return _tool_result(tool_call, tool_name,
                            {"success": False, "error": f"Unknown tool: {tool_name}"})

    try:
        if tool_name == "create_task":
            result = service.create_task(
                arguments["title"], date.fromisoformat(arguments["due_date"]),
                due_time=arguments.get("due_time"), due_period=arguments.get("due_period"))
        elif tool_name == "list_pending_tasks":
            result = service.list_pending_tasks(
                due_date=date.fromisoformat(arguments["due_date"])
                if arguments.get("due_date") else None,
                end_date=date.fromisoformat(arguments["end_date"])
                if arguments.get("end_date") else None,
                due_time=arguments.get("due_time"),
                due_period=arguments.get("due_period"),
                limit=arguments.get("limit"))
        elif tool_name == "complete_tasks_by_date":
            result = service.prepare_bulk_completion(
                date.fromisoformat(arguments["due_date"]),
                end_date=date.fromisoformat(arguments["end_date"])
                if arguments.get("end_date") else None)
        else:
            result = getattr(service, handler_name)(arguments["task_query"])
    except Exception as exc:
        return _tool_result(tool_call, tool_name,
                            {"success": False, "error": f"Execution error: {exc}"})

    payload = {"success": result.success, "status": result.status, "message": result.message}
    if result.task:
        payload["task"] = {"id": result.task.id, "title": result.task.title}
    if result.tasks:
        payload["tasks"] = [
            {"id": t.id, "title": t.title, "due_date": t.due_date.isoformat(),
             "due_time": t.due_time, "due_period": t.due_period, "status": t.status}
            for t in result.tasks
        ]
    if result.matches:
        payload["matches"] = [
            {"id": t.id, "title": t.title, "due_date": t.due_date.isoformat(),
             "due_time": t.due_time, "due_period": t.due_period, "status": t.status}
            for t in result.matches
        ]
    return _tool_result(tool_call, tool_name, payload)


# --- Destructive-request policy ----------------------------------------------
# Deleting or clearing tasks is not implemented anywhere. Requests are refused
# deterministically BEFORE any model call, and the model cannot invent a
# deletion tool because the allowlist only contains the five operations above.

DESTRUCTIVE_OPERATION_REFUSAL = (
    "I can't delete tasks. Bulk deletion is not supported, so nothing was "
    "deleted. Tasks can only be created, listed, updated to completed, or "
    "viewed. To finish tasks, say for example \"Mark all tomorrow's tasks as "
    "done\" (you will be asked to confirm) or \"Mark the report as completed\"."
)

_DESTRUCTIVE_VERBS = ("clear", "delete", "remove", "erase", "wipe", "purge", "drop")


def is_destructive_task_intent(message):
    if not message:
        return False
    lowered = message.lower()
    verb_pattern = r"\b(?:" + "|".join(_DESTRUCTIVE_VERBS) + r")\b"
    if not re.search(verb_pattern, lowered):
        return False
    return bool(re.search(verb_pattern + r"(?:\s+\S+){0,3}?\s+tasks?\b", lowered))


print(f"Exposed tools: {', '.join(t['function']['name'] for t in TOOLS)}")
print("Any other tool name is rejected by the allowlist; deletion has no tool at all.")

Exposed tools: create_task, list_pending_tasks, complete_task, get_task_details, complete_tasks_by_date
Any other tool name is rejected by the allowlist; deletion has no tool at all.


### Step 6 — Offline mock LLM (deterministic natural-language understanding)

A real LLM decides which tool to call; the mock does the same with simple rules that cover the
required scenarios. The response shape is identical to a function-calling provider, so the rest
of the pipeline cannot tell the difference.

In [6]:
_DATE_RE = re.compile(
    r"\b(\d{4}-\d{2}-\d{2})\b"
    r"|\b(tomorrow|today|tonight|tmrw)\b"
    r"|\b(next\s+week)\b"
    r"|\b((?:next|this)\s+(?:monday|tuesday|wednesday|thursday|friday|saturday|sunday))\b"
    r"|\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b"
    r"|\b(in\s+\d{1,3}\s+days?)\b",
    re.IGNORECASE,
)
_TIME_12_RE = re.compile(r"\b(\d{1,2})(?::(\d{2}))?\s*([ap])\.?m\.?\b", re.IGNORECASE)
_TIME_24_RE = re.compile(r"\b(\d{1,2}):(\d{2})\b")
_TIME_PERIOD_RE = re.compile(r"\b(morning|afternoon|evening|night)\b", re.IGNORECASE)
_AMBIGUOUS_AT_RE = re.compile(r"\bat\s+(\d{1,2})\b", re.IGNORECASE)
_WEEKDAYS = {name: i for i, name in enumerate(
    ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"])}


def _find_dates(text):
    return [(m.group(0), m.start(), m.end()) for m in _DATE_RE.finditer(text)]


def _extract_time_and_period(text):
    """Return (due_time, due_period, remaining_text, ambiguous_hour)."""
    due_time = due_period = ambiguous_hour = None
    working = text

    match = _TIME_12_RE.search(working)
    if match:
        hour, minute = int(match.group(1)), int(match.group(2) or 0)
        due_time = normalize_time_value(f"{hour}:{minute:02d} {match.group(3)}M")
        working = (working[:match.start()] + " " + working[match.end():]).strip()
    else:
        match = _TIME_24_RE.search(working)
        if match:
            due_time = normalize_time_value(match.group(0))
            working = (working[:match.start()] + " " + working[match.end():]).strip()
        else:
            match = _AMBIGUOUS_AT_RE.search(working)
            if match:
                hour = int(match.group(1))
                if 13 <= hour <= 23:      # "at 15" is unambiguous
                    due_time = f"{hour:02d}:00"
                    working = (working[:match.start()] + " " + working[match.end():]).strip()
                elif 1 <= hour <= 12:     # "at 3" needs an AM/PM clarification
                    ambiguous_hour = str(hour)

    if due_time is None and ambiguous_hour is None:
        match = _TIME_PERIOD_RE.search(working)
        if match:
            due_period = match.group(1).lower()
            working = (working[:match.start()] + " " + working[match.end():]).strip()

    return due_time, due_period, re.sub(r"\s+", " ", working).strip(), ambiguous_hour


def _clean_title(value):
    value = value.strip().strip(" .,!?:;")
    value = re.sub(r"^(?:the|my|a|an)\s+", "", value, flags=re.IGNORECASE)
    value = re.sub(r"(?:\s+(?:at|on|by|for|to|due))+$", "", value, flags=re.IGNORECASE)
    return value.strip().strip(" .,!?:;")


def _parse_date(expression, today):
    """Resolve common absolute/relative date expressions to a date."""
    if not expression:
        return None
    value = expression.lower().strip().strip(" .,!?:;")
    iso = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", value)
    if iso:
        try:
            return date.fromisoformat(iso.group(1))
        except ValueError:
            return None
    if value in {"today", "now", "tonight"}:
        return today
    if value in {"tomorrow", "tmrw"}:
        return today + timedelta(days=1)
    if value in {"next week", "in one week", "in a week"}:
        return today + timedelta(days=7)
    in_days = re.search(r"\bin\s+(\d{1,3})\s+days?\b", value)
    if in_days:
        return today + timedelta(days=int(in_days.group(1)))
    weekday = re.search(
        r"\b(?:(next|this)\s+)?(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b",
        value,
    )
    if weekday:
        qualifier, target = weekday.group(1), _WEEKDAYS[weekday.group(2)]
        delta = (target - today.weekday()) % 7
        if qualifier == "next":
            delta = delta or 7
        elif qualifier is None:
            delta = delta or 7
        return today + timedelta(days=delta)
    return None


class MockLLMClient:
    """Deterministic offline stand-in for a function-calling LLM."""

    def __init__(self, today=None):
        self.today = today or TODAY
        self.call_count = 0

    def get_model_name(self):
        return "mock-llm (offline)"

    def chat(self, messages, tools, tool_choice="auto"):
        self.call_count += 1
        raw = ""
        for message in reversed(messages):
            if message.get("role") == "user":
                raw = message.get("content", "").strip()
                break
        lowered = raw.lower()
        today = self.today

        # 0) Never even consider a delete/clear operation.
        if is_destructive_task_intent(raw):
            return {"content": DESTRUCTIVE_OPERATION_REFUSAL, "tool_calls": None}

        # 1) Details: "When is X due?", "details of X"
        if any(word in lowered for word in ("detail", "when", "due date", "status")):
            match = (re.search(r"when\s+is\s+(.+?)\s+due", raw, re.IGNORECASE)
                     or re.search(r"(?:details?|status)\s+(?:of|for)\s+(.+)", raw, re.IGNORECASE)
                     or re.search(r"when\s+is\s+(.+)", raw, re.IGNORECASE))
            if match:
                return self._tool("get_task_details", {"task_query": _clean_title(match.group(1))})

        # 2) Bulk completion by date (checked before single-task completion)
        if (re.search(r"\b(complete|completed|mark|finish|done)\b", lowered)
                and (re.search(r"\b(all|every|both)\b", lowered) or "tasks" in lowered)):
            dates = _find_dates(raw)
            if not dates:
                return {"content": ("Which date's tasks should I mark as completed? "
                                    "Please provide a date (for example 'tomorrow' or "
                                    "'2026-10-20')."),
                        "tool_calls": None}
            due = _parse_date(dates[0][0], today)
            if due is None:
                return {"content": (f"I couldn't understand the date '{dates[0][0]}'. "
                                    "Please use a concrete date such as 2026-10-20."),
                        "tool_calls": None}
            args = {"due_date": due.isoformat()}
            if len(dates) > 1:
                end = _parse_date(dates[1][0], today)
                if end and end >= due:
                    args["end_date"] = end.isoformat()
            return self._tool("complete_tasks_by_date", args)

        # 3) List / schedule / next task
        if any(marker in lowered for marker in
               ("pending", "list", "schedule", "what do i have", "next task",
                "upcoming", "my day")):
            due_time, due_period, _, ambiguous = _extract_time_and_period(raw)
            if ambiguous:
                return self._ask_time(ambiguous)
            args = {}
            dates = _find_dates(raw)
            if dates:
                parsed = _parse_date(dates[0][0], today)
                if parsed is None:
                    return {"content": (f"I couldn't understand the date '{dates[0][0]}'. "
                                        "Please use a concrete date such as 2026-10-20."),
                            "tool_calls": None}
                args["due_date"] = parsed.isoformat()
            if due_time:
                args["due_time"] = due_time
            elif due_period:
                args["due_period"] = due_period
            if "next task" in lowered:
                args["limit"] = 1
            return self._tool("list_pending_tasks", args)

        # 4) Create: "add/create a task ..." or a short imperative like
        #    "Train at 3 PM on Saturday"
        if ((re.search(r"\b(add|create|new)\b", lowered) and "task" in lowered)
                or self._bare_create(raw)):
            title, date_expression, due_time, due_period, ambiguous = self._extract_create(raw)
            if ambiguous:
                return self._ask_time(ambiguous)
            due = _parse_date(date_expression, today)
            if due is None:
                return {"content": (f"What due date should I use for '{title}'? "
                                    "Please provide a date (YYYY-MM-DD) or say 'tomorrow'."),
                        "tool_calls": None}
            args = {"title": title or "Mock task", "due_date": due.isoformat()}
            if due_time:
                args["due_time"] = due_time
            elif due_period:
                args["due_period"] = due_period
            return self._tool("create_task", args)

        # 5) Single-task completion
        if any(word in lowered for word in ("complete", "mark", "finish")):
            match = re.search(r"(?:complete|mark|finish)\s+(.+)$", raw, re.IGNORECASE)
            if match:
                raw_query = re.sub(
                    r"\s+as\s+(?:completed|complete|done)[.!?]*\s*$", "",
                    match.group(1), flags=re.IGNORECASE,
                )
                query = _clean_title(raw_query)
                query = re.sub(r"^task\s+", "", query, flags=re.IGNORECASE).strip()
                if query:
                    return self._tool("complete_task", {"task_query": query})

        return {"content": ("I can help you manage tasks. Try asking me to create a task, "
                            "list pending tasks, complete a task, or get task details."),
                "tool_calls": None}

    # -- helpers -------------------------------------------------------------
    @staticmethod
    def _tool(name, arguments):
        return {"content": None, "tool_calls": [{
            "id": f"call_{name}", "type": "function",
            "function": {"name": name, "arguments": json.dumps(arguments)}}]}

    @staticmethod
    def _ask_time(hour):
        return {"content": (f"Did you mean {hour} AM or {hour} PM? "
                            "Please provide an exact time such as '3 PM' or '15:00'."),
                "tool_calls": None}

    @staticmethod
    def _bare_create(message):
        lowered = message.lower().strip()
        if lowered.endswith("?"):
            return False
        if any(marker in lowered for marker in
               ("show", "list", "pending", "schedule", "complete", "mark",
                "finish", "done", "when", "what", "details", "status")):
            return False
        has_time = bool(_TIME_12_RE.search(lowered) or _TIME_24_RE.search(lowered)
                        or _TIME_PERIOD_RE.search(lowered))
        return has_time and bool(_find_dates(lowered))

    @staticmethod
    def _extract_create(message):
        """Return (title, date_expression, due_time, due_period, ambiguous_hour)."""
        rest = message
        prefix = re.search(
            r"^(?:please\s+)?(?:add|create|new)\s+(?:a\s+)?(?:new\s+)?task\b\s*"
            r"(?:to\s+|:)?\s*(.*)$",
            rest, flags=re.IGNORECASE,
        )
        if prefix:
            rest = prefix.group(1).strip()
        else:
            rest = re.sub(r"^(?:please\s+)?(?:add|create)\s+", "", rest,
                          flags=re.IGNORECASE).strip()
        if not rest:
            return "Mock task", None, None, None, None

        due_time, due_period, rest, ambiguous = _extract_time_and_period(rest)
        if ambiguous:
            return None, None, None, None, ambiguous

        date_expression = None
        date_matches = _find_dates(rest)
        if date_matches:
            for expression, start, end in reversed(date_matches):
                rest = (rest[:start] + " " + rest[end:]).strip()
            date_expression = date_matches[0][0]
        else:
            # The date may sit after a time phrase that was already removed.
            for expression, _, _ in _find_dates(message):
                if re.search(re.escape(expression), rest, re.IGNORECASE):
                    rest = re.sub(re.escape(expression), " ", rest, count=1,
                                  flags=re.IGNORECASE).strip()
                    date_expression = expression
                    break
        return _clean_title(rest) or "Mock task", date_expression, due_time, due_period, None


print("MockLLMClient ready: deterministic rule-based NLU with the real tool-call shape")

MockLLMClient ready: deterministic rule-based NLU with the real tool-call shape


### Step 7 — Assistant orchestration (clarification, confirmation, safety)

The assistant calls the LLM, executes the returned tool call, and manages two pieces of
conversation state: a pending clarification (ambiguous task reference) and a pending
bulk-completion confirmation. Destructive requests are refused before any model call.

In [7]:
class Assistant:
    _AFFIRMATIVE = {"yes", "y", "yeah", "yep", "confirm", "confirmed", "ok",
                    "okay", "sure", "proceed", "please"}
    _NEGATIVE = {"no", "n", "nope", "cancel", "stop", "abort", "nevermind"}

    def __init__(self, service=SERVICE, llm=None):
        self.service = service
        self.llm = llm or MockLLMClient()
        self._pending_clarification = None
        self._pending_confirmation = None

    def process_message(self, user_message):
        if not user_message or not user_message.strip():
            return "Please provide a message."

        # A pending bulk-completion confirmation is answered first.
        if self._pending_confirmation:
            return self._handle_confirmation(user_message.strip())

        # Then a pending clarification of an ambiguous task reference.
        if self._pending_clarification:
            return self._handle_clarification(user_message.strip())

        # Hard safety guard: deletion requests never reach the model.
        if is_destructive_task_intent(user_message):
            return DESTRUCTIVE_OPERATION_REFUSAL

        try:
            response = self.llm.chat([{"role": "user", "content": user_message.strip()}], TOOLS)
        except RetryExhaustedError as exc:
            return str(exc)
        except Exception as exc:
            return f"An unexpected error occurred: {exc}"

        if response.get("tool_calls"):
            return self._handle_tool_calls(response["tool_calls"])
        return response.get("content") or "I'm not sure how to help with that."

    @property
    def has_pending_confirmation(self):
        return self._pending_confirmation is not None

    @property
    def has_pending_clarification(self):
        return self._pending_clarification is not None

    # -- tool execution --------------------------------------------------------
    def _handle_tool_calls(self, tool_calls):
        last_payload, confirmation_message = None, None
        for tool_call in tool_calls:
            result = execute_tool_call(tool_call, self.service)
            payload = json.loads(result["content"])
            last_payload = payload

            if payload.get("status") == "confirmation_required" and payload.get("tasks"):
                self._pending_confirmation = {
                    "tool_name": tool_call["function"]["name"],
                    "ids": [t["id"] for t in payload["tasks"]],
                }
                confirmation_message = payload.get("message")
                continue

            if payload.get("status") == "ambiguous" and payload.get("matches"):
                self._pending_clarification = {
                    "tool_name": tool_call["function"]["name"],
                    "matches": payload["matches"],
                }

        if confirmation_message:
            return confirmation_message

        if self._pending_clarification:
            lines = [
                f"  {i}. {m['title']} "
                f"(due: {format_due(m['due_date'], m.get('due_time'), m.get('due_period'))}, "
                f"status: {m['status']})"
                for i, m in enumerate(self._pending_clarification["matches"], 1)
            ]
            return ("I found multiple matching tasks:\n" + "\n".join(lines)
                    + "\n\nPlease specify which task you mean "
                      "(e.g., 'the first one', 'task 2', or the full title).")

        # Never fall back to a success-sounding message when the tool produced
        # no usable result.
        if isinstance(last_payload, dict):
            if last_payload.get("message"):
                return last_payload["message"]
            if last_payload.get("error"):
                return f"Request failed: {last_payload['error']}"
        return ("I couldn't process the tool result safely, so no task changes "
                "were reported. Please try again.")

    # -- bulk confirmation -------------------------------------------------------
    def _handle_confirmation(self, user_response):
        pending = self._pending_confirmation

        # A destructive message cancels the confirmation and is refused.
        if is_destructive_task_intent(user_response):
            self._pending_confirmation = None
            return DESTRUCTIVE_OPERATION_REFUSAL

        normalized = user_response.strip().lower().strip(" .,!?:;")
        first_word = re.split(r"[^a-z0-9]+", normalized)[0] if normalized else ""

        if normalized in self._AFFIRMATIVE or first_word in self._AFFIRMATIVE:
            self._pending_confirmation = None
            return self.service.complete_tasks_by_ids(pending["ids"]).message

        if normalized in self._NEGATIVE or first_word in self._NEGATIVE:
            self._pending_confirmation = None
            return "Bulk completion cancelled. No tasks were changed."

        # Abandon the confirmation if the user starts a new command instead.
        new_intent_markers = ("add ", "create ", "new task", "show ", "list ", "pending",
                              "schedule", "next task", "complete ", "mark ", "finish ",
                              "when ", "details", "due ")
        if any(marker in normalized for marker in new_intent_markers):
            self._pending_confirmation = None
            return self.process_message(user_response)

        return "Please answer 'yes' to confirm the bulk completion or 'no' to cancel."

    # -- ambiguity clarification --------------------------------------------------
    def _handle_clarification(self, user_response):
        if is_destructive_task_intent(user_response):
            self._pending_clarification = None
            return DESTRUCTIVE_OPERATION_REFUSAL

        matches = self._pending_clarification["matches"]
        tool_name = self._pending_clarification["tool_name"]
        selected = self._resolve_clarification(user_response, matches)
        if selected is None:
            # A brand-new command abandons the old clarification.
            lowered = user_response.lower()
            if any(marker in lowered for marker in
                   ("add ", "create ", "new task", "show ", "list ", "pending",
                    "complete ", "mark ", "finish ", "when ", "details", "due ")):
                self._pending_clarification = None
                return self.process_message(user_response)
            return ("I couldn't understand which task you meant. "
                    "Please specify the number (e.g., '1', '2') or the full task title.")

        self._pending_clarification = None
        if tool_name == "complete_task":
            return self.service.complete_task(selected["title"]).message
        if tool_name == "get_task_details":
            return self.service.get_task_details(selected["title"]).message
        return "Clarification not supported for this operation."

    @staticmethod
    def _resolve_clarification(response, matches):
        text = response.lower().strip()
        try:
            index = int(text) - 1
            if 0 <= index < len(matches):
                return matches[index]
        except ValueError:
            pass
        for match in matches:
            if match["title"].lower() == text:
                return match
        for match in matches:
            if text in match["title"].lower() or match["title"].lower() in text:
                return match
        for word, index in {"first": 0, "second": 1, "third": 2, "last": -1}.items():
            if word in text and matches:
                return matches[index]
        return None


print("Assistant ready: mock LLM + validated tools + confirmation/clarification state")

Assistant ready: mock LLM + validated tools + confirmation/clarification state


### Step 8 — Transient-error handling (bounded retries)

The GitHub project retries Gemini HTTP **429/503** errors up to a configured number of times with
exponential backoff (`base_delay * 2**attempt`); timeouts and HTTP 400 are never retried, and an
exhausted retry produces a friendly message stating that no task changes were made. The same
policy is implemented here. The demonstrations use **stubs**, so no network call is made.

In [8]:
RETRYABLE_STATUS_CODES = frozenset({429, 503})
_TRANSIENT_TEXT_RE = re.compile(
    r"(?i)\b(429|503|resource[_ ]?exhausted|service[_ ]?unavailable|"
    r"unavailable|overloaded|high demand|try again later)\b")


class RetryExhaustedError(Exception):
    """Raised after bounded retries are exhausted (friendly, redacted message)."""


def is_retryable_error(exc):
    """Numeric status codes are authoritative; text matching is only a fallback."""
    for attribute in ("code", "status_code"):
        value = getattr(exc, attribute, None)
        if isinstance(value, int) and not isinstance(value, bool):
            return value in RETRYABLE_STATUS_CODES
    return bool(_TRANSIENT_TEXT_RE.search(str(exc)))


def retry_with_backoff(call, max_retries=3, base_delay=0.5, sleep=time.sleep):
    """Call ``call()``, retrying transient 429/503 errors with exponential backoff."""
    attempt = 0
    while True:
        try:
            return call()
        except Exception as exc:
            if not is_retryable_error(exc):
                raise
            if attempt >= max_retries:
                raise RetryExhaustedError(
                    f"The assistant is temporarily unavailable. I retried "
                    f"{max_retries} time(s) with exponential backoff without success. "
                    "No task changes were made. Please wait a moment and try again."
                ) from exc
            sleep(base_delay * (2 ** attempt))
            attempt += 1


# --- Offline stubs used only for demonstrations (no real API calls) ----------
class StubAPIError(Exception):
    """Stands in for a Gemini HTTP error; ``code`` mirrors the SDK's exc.code."""
    def __init__(self, code, message="simulated API error"):
        super().__init__(message)
        self.code = code


class FailingLLMClient:
    """Stub LLM that always behaves like Gemini being unavailable (simulation)."""
    def __init__(self, code=503):
        self.code = code

    def get_model_name(self):
        return f"stub-failing-llm ({self.code})"

    def chat(self, messages, tools, tool_choice="auto"):
        raise RetryExhaustedError(
            "The assistant is temporarily unavailable. I retried 3 time(s) with "
            "exponential backoff without success. No task changes were made. "
            "Please wait a moment and try again."
        )


print("Retry policy ready: retryable = HTTP 429/503 only; delay = base * 2**attempt")
print("Stub clients defined for offline error demonstrations (no network).")

Retry policy ready: retryable = HTTP 429/503 only; delay = base * 2**attempt
Stub clients defined for offline error demonstrations (no network).


### Step 9 — Optional live Gemini integration (not executed in this notebook)

All demonstrations above used the offline mock. This section is **optional**: it defines a live
client with the same tool contract, but it deliberately does **not** call the API in the saved
outputs. The key is read from the environment variable `GOOGLE_API_KEY` and is never written
into the notebook.

To try live mode yourself:

```bash
export GOOGLE_API_KEY="your-key"        # never commit or print the key
pip install google-genai                # the SDK used by the GitHub project
```

Then run `Assistant(llm=GeminiLLMClient()).process_message(...)` manually. Any real response is
a **live Gemini** answer, clearly different from the mock demonstrations above.

In [9]:
# OPTIONAL — this cell defines the live adapter but makes NO API call.
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY", "")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")

try:
    import google.genai  # noqa: F401
    GEMINI_SDK_AVAILABLE = True
except ImportError:
    GEMINI_SDK_AVAILABLE = False

LIVE_READY = bool(GEMINI_API_KEY) and GEMINI_SDK_AVAILABLE
print(f"Gemini SDK installed: {GEMINI_SDK_AVAILABLE}")
print(f"GOOGLE_API_KEY present: {bool(GEMINI_API_KEY)}")
print(f"Live mode ready: {LIVE_READY}")
print("Saved outputs contain no API key and no live API call.")


class GeminiLLMClient:
    """Optional live client with the same contract as MockLLMClient.

    Mirrors the GitHub project's Gemini adapter: shared tool schemas, bounded
    retries for 429/503, and no secrets in messages or errors.
    """

    def __init__(self, model=GEMINI_MODEL, max_retries=3, retry_base_delay=0.5):
        if not os.getenv("GOOGLE_API_KEY"):
            raise ValueError("Set GOOGLE_API_KEY in the environment to use live mode")
        from google import genai
        from google.genai import types
        self._types = types
        self._client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.model = model
        self.max_retries = max_retries
        self.retry_base_delay = retry_base_delay

    def get_model_name(self):
        return self.model

    def _to_schema(self, schema):
        types = self._types
        kwargs = {}
        if schema.get("description"):
            kwargs["description"] = schema["description"]
        if schema.get("enum"):
            kwargs["enum"] = [str(item) for item in schema["enum"]]
        schema_type = str(schema.get("type", "string")).lower()
        if schema_type == "object":
            kwargs["type"] = types.Type.OBJECT
            if schema.get("properties"):
                kwargs["properties"] = {
                    key: self._to_schema(value)
                    for key, value in schema["properties"].items()
                }
            if schema.get("required"):
                kwargs["required"] = list(schema["required"])
        elif schema_type == "array":
            kwargs["type"] = types.Type.ARRAY
            if schema.get("items"):
                kwargs["items"] = self._to_schema(schema["items"])
        else:
            type_map = {"string": "STRING", "number": "NUMBER", "integer": "INTEGER",
                        "boolean": "BOOLEAN", "array": "ARRAY", "object": "OBJECT",
                        "null": "NULL"}
            kwargs["type"] = getattr(types.Type, type_map.get(schema_type, "STRING"))
        return types.Schema(**kwargs)

    def _build_tools(self, tools):
        types = self._types
        declarations = []
        for tool in tools:
            function = tool["function"]
            declarations.append(types.FunctionDeclaration(
                name=function["name"],
                description=function.get("description", ""),
                parameters=self._to_schema(
                    function.get("parameters", {"type": "object", "properties": {}})
                ),
            ))
        return [types.Tool(function_declarations=declarations)]

    def _parse_response(self, response):
        candidates = getattr(response, "candidates", None)
        if not candidates:
            raise RuntimeError("Gemini returned no candidates")
        content = getattr(candidates[0], "content", None)
        text_chunks, tool_calls = [], []
        for index, part in enumerate(getattr(content, "parts", None) or []):
            if getattr(part, "thought", False):
                continue
            call = getattr(part, "function_call", None)
            if call is not None:
                arguments = dict(call.args) if getattr(call, "args", None) else {}
                tool_calls.append({
                    "id": getattr(call, "id", None) or f"call_{index + 1}",
                    "type": "function",
                    "function": {"name": call.name, "arguments": json.dumps(arguments)},
                })
            elif getattr(part, "text", None):
                text_chunks.append(part.text)
        return {"content": "".join(text_chunks) or None, "tool_calls": tool_calls or None}

    def chat(self, messages, tools, tool_choice="auto"):
        types = self._types
        contents = [
            types.Content(role="user", parts=[types.Part(text=m.get("content") or "")])
            for m in messages if m.get("role") == "user"
        ] or [types.Content(role="user", parts=[types.Part(text="")])]
        config = types.GenerateContentConfig(tools=self._build_tools(tools))
        response = retry_with_backoff(
            lambda: self._client.models.generate_content(
                model=self.model, contents=contents, config=config),
            max_retries=self.max_retries,
            base_delay=self.retry_base_delay,
        )
        return self._parse_response(response)


# Example usage (commented out on purpose — running it would make a live call):
# if LIVE_READY:
#     live = Assistant(llm=GeminiLLMClient())
#     print(live.process_message("Add a task to prepare the report by tomorrow"))

Gemini SDK installed: True
GOOGLE_API_KEY present: False
Live mode ready: False
Saved outputs contain no API key and no live API call.


## 5. Sample natural-language inputs and outputs (offline mock mode)

Everything in this section runs against the **mock** client — no credentials, no network. Each
reply below is generated by the notebook's own code from the actual database state.

In [10]:
def show(message, response):
    print(f"User: {message}")
    print(f"Assistant: {response}")
    print()


assistant = Assistant()   # fresh conversation state

print("=== Creating tasks from natural language ===")
show("Add a task to prepare for the GenAI assignment by tomorrow",
     assistant.process_message("Add a task to prepare for the GenAI assignment by tomorrow"))
show("Add a task to call mom tomorrow morning",
     assistant.process_message("Add a task to call mom tomorrow morning"))
show("Train at 3 PM on Saturday",
     assistant.process_message("Train at 3 PM on Saturday"))
show("Add a task to water the plants by tomorrow",
     assistant.process_message("Add a task to water the plants by tomorrow"))

stored = [t for t in SERVICE.repo.get_pending_tasks_filtered() if t.title == "water the plants"][0]
print(f"Stored without a time: due_date={stored.due_date.isoformat()}, "
      f"due_time={stored.due_time!r}, due_period={stored.due_period!r}")
print()

show("Add a task to read a book",     # no date given: the assistant asks instead of guessing
     assistant.process_message("Add a task to read a book"))

print("=== Listing and filtering pending tasks ===")
show("Show me all my pending tasks", assistant.process_message("Show me all my pending tasks"))
show("Show my schedule for tomorrow", assistant.process_message("Show my schedule for tomorrow"))
show("What is my next task?", assistant.process_message("What is my next task?"))

print("=== Task details and single-task completion ===")
show("When is the GenAI assignment due?",
     assistant.process_message("When is the GenAI assignment due?"))
show("Mark the GenAI assignment as completed",
     assistant.process_message("Mark the GenAI assignment as completed"))
show("Mark the GenAI assignment as completed",   # repeat -> accurate "already completed"
     assistant.process_message("Mark the GenAI assignment as completed"))

=== Creating tasks from natural language ===
User: Add a task to prepare for the GenAI assignment by tomorrow
Assistant: Task created: prepare for the GenAI assignment (due: 2026-10-20, status: pending)

User: Add a task to call mom tomorrow morning
Assistant: Task created: call mom (due: 2026-10-20 (morning), status: pending)

User: Train at 3 PM on Saturday
Assistant: Task created: Train (due: 2026-10-24 at 3:00 PM, status: pending)

User: Add a task to water the plants by tomorrow
Assistant: Task created: water the plants (due: 2026-10-20, status: pending)

Stored without a time: due_date=2026-10-20, due_time=None, due_period=None

User: Add a task to read a book
Assistant: What due date should I use for 'read a book'? Please provide a date (YYYY-MM-DD) or say 'tomorrow'.

=== Listing and filtering pending tasks ===
User: Show me all my pending tasks
Assistant: Pending tasks (4):
  [1] prepare for the GenAI assignment — due: 2026-10-20 — status: pending
  [2] call mom — due: 2026-10

In [11]:
print("=== Ambiguous time is clarified, never invented ===")
show("Add a task to jog at 3", assistant.process_message("Add a task to jog at 3"))
show("Add a task to jog at 3 PM tomorrow",
     assistant.process_message("Add a task to jog at 3 PM tomorrow"))

print("=== Bulk completion requires an explicit confirmation ===")
show("Mark all tomorrow's tasks as done",
     assistant.process_message("Mark all tomorrow's tasks as done"))
print("Pending tasks while waiting for confirmation:",
      len(SERVICE.repo.get_pending_tasks_filtered()))
show("yes", assistant.process_message("yes"))
print("Pending tasks after confirmation:",
      len(SERVICE.repo.get_pending_tasks_filtered()))
print()

print("=== Safe cancellation changes nothing ===")
assistant.process_message("Add a task to pack for the trip by tomorrow")
pending_before = SERVICE.repo.count_tasks("pending")
show("Mark all tomorrow's tasks as done",
     assistant.process_message("Mark all tomorrow's tasks as done"))
show("no", assistant.process_message("no"))
print(f"Pending before cancellation: {pending_before} | after: "
      f"{SERVICE.repo.count_tasks('pending')}")
print()

print("=== Ambiguous task reference asks which task is meant ===")
assistant.process_message("Add a task to finish the report by tomorrow")
time.sleep(0.01)   # keep creation timestamps distinct so the order is deterministic
assistant.process_message("Add a task to email the report by Saturday")
show("Mark the report as completed", assistant.process_message("Mark the report as completed"))
show("2", assistant.process_message("2"))
print("Remaining pending tasks:", SERVICE.repo.count_tasks("pending"))
print()

print("=== Bulk deletion is refused; nothing changes ===")
total_before = SERVICE.repo.count_tasks()
show("Delete all my tasks", assistant.process_message("Delete all my tasks"))
print(f"Total tasks before: {total_before} | after: {SERVICE.repo.count_tasks()}")
print()

print("=== A task that does not exist ===")
show("Mark the skydiving lesson as completed",
     assistant.process_message("Mark the skydiving lesson as completed"))

=== Ambiguous time is clarified, never invented ===
User: Add a task to jog at 3
Assistant: Did you mean 3 AM or 3 PM? Please provide an exact time such as '3 PM' or '15:00'.

User: Add a task to jog at 3 PM tomorrow
Assistant: Task created: jog (due: 2026-10-20 at 3:00 PM, status: pending)

=== Bulk completion requires an explicit confirmation ===
User: Mark all tomorrow's tasks as done
Assistant: I found 3 pending task(s) for 2026-10-20:
  1. [id 2] call mom — due: 2026-10-20 (morning)
  2. [id 4] water the plants — due: 2026-10-20
  3. [id 5] jog — due: 2026-10-20 at 3:00 PM
Should I mark all 3 as completed? (yes/no)

Pending tasks while waiting for confirmation: 4
User: yes
Assistant: Completed 3 task(s): call mom, water the plants, jog.

Pending tasks after confirmation: 1

=== Safe cancellation changes nothing ===
User: Mark all tomorrow's tasks as done
Assistant: I found 1 pending task(s) for 2026-10-20:
  1. [id 6] pack for the trip — due: 2026-10-20
Should I mark all 1 as comp

## 6. Testing core functionality and error handling

The checks below are a **compact, self-contained test suite for this notebook**. They are *not*
the GitHub project's 312-test pytest suite; everything runs in this file against temporary
SQLite databases. Each check prints PASS or FAIL, and the notebook stops with an error if any
check fails.

In [12]:
TOTAL_CHECKS = []
FAILED = []


def check(name, condition, detail=""):
    ok = bool(condition)
    TOTAL_CHECKS.append(name)
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
    if not ok:
        FAILED.append((name, detail))


def _raises(call):
    """Return the exception raised by ``call``, or None."""
    try:
        call()
    except Exception as exc:
        return exc
    return None


def fresh_service():
    return TaskService(TaskRepository(WORKDIR / f"test_{len(TOTAL_CHECKS)}.db"))


print("=== Core functionality ===")

# --- date/time normalization ---
check("12-hour time '3 PM' normalizes to 15:00", normalize_time_value("3 PM") == "15:00")
check("'3:30 PM' normalizes to 15:30", normalize_time_value("3:30 PM") == "15:30")
check("'15:00' stays 15:00", normalize_time_value("15:00") == "15:00")
check("'midnight' normalizes to 00:00", normalize_time_value("midnight") == "00:00")
check("'Morning.' normalizes to morning", normalize_period_value("Morning.") == "morning")
check("invalid time '25:00' is rejected",
      isinstance(_raises(lambda: normalize_time_value("25:00")), ValueError))
check("unknown period 'dawn' is rejected",
      isinstance(_raises(lambda: normalize_period_value("dawn")), ValueError))

# --- storage and service behavior ---
service = fresh_service()
untimed = service.create_task("submit the tax form", TOMORROW)
check("creating a task succeeds", untimed.success and untimed.task.title == "submit the tax form")
check("a task without a time stores NULL time and NULL period",
      untimed.task.due_time is None and untimed.task.due_period is None)

timed = service.create_task("Standup", TOMORROW, due_time="10:00")
check("an exact time is stored as HH:MM", timed.task.due_time == "10:00")
check("display shows a readable 12-hour time",
      timed.task.due_display.endswith("at 10:00 AM"))

periodic = service.create_task("Call mom", TOMORROW, due_period="evening")
check("a period is stored separately from time",
      periodic.task.due_period == "evening" and periodic.task.due_time is None)
check("display shows the approximate period",
      periodic.task.due_display.endswith("(evening)"))

both = service.create_task("Bad task", TOMORROW, due_time="10:00", due_period="evening")
check("time + period together is rejected", (not both.success) and "not both" in both.message)
check("an empty title is rejected", not service.create_task("   ", TOMORROW).success)

listed = service.list_pending_tasks(due_date=TOMORROW)
check("date filter returns exactly that day's pending tasks", len(listed.tasks) == 3)
check("ordering: untimed tasks first, then exact times",
      [t.due_time for t in listed.tasks] == [None, None, "10:00"])
check("period filter works",
      [t.title for t in service.list_pending_tasks(due_period="evening").tasks] == ["Call mom"])
check("limit=1 returns the next task", len(service.list_pending_tasks(limit=1).tasks) == 1)

check("details by partial title works", service.get_task_details("tax").success)
completed = service.complete_task("Standup")
check("single-task completion succeeds",
      completed.success and completed.task.status == "completed")
check("re-completing reports 'already completed'",
      service.complete_task("Standup").status == "already_completed")
check("a nonexistent task reports not_found",
      service.complete_task("skydiving lesson").status == "not_found")

# --- ambiguity ---
dup = fresh_service()
dup.create_task("finish the report", TOMORROW)
dup.create_task("email the report", TOMORROW)
ambiguous = dup.complete_task("report")
check("an ambiguous title returns a clarification instead of a guess",
      ambiguous.status == "ambiguous" and len(ambiguous.matches) == 2)

# --- mock natural-language understanding ---
mock_service = fresh_service()
mock_assistant = Assistant(service=mock_service, llm=MockLLMClient())
mock_assistant.process_message("Add a task to call mom tomorrow morning")
task = mock_service.repo.get_pending_tasks_filtered()[0]
check("mock extracts title, date, and period from natural language",
      task.title == "call mom" and task.due_date == TOMORROW
      and task.due_period == "morning" and task.due_time is None)

mock_assistant.process_message("Train at 3 PM on Saturday")
task = [t for t in mock_service.repo.get_pending_tasks_filtered() if t.title == "Train"][0]
check("mock extracts an exact time and keeps the title clean",
      task.due_time == "15:00" and task.due_date >= TOMORROW)

reply = mock_assistant.process_message("Add a task to jog at 3")
check("mock asks AM/PM for an ambiguous 'at 3' instead of inventing a time",
      "3 AM or 3 PM" in reply and mock_service.repo.count_tasks() == 2)

=== Core functionality ===
PASS  12-hour time '3 PM' normalizes to 15:00
PASS  '3:30 PM' normalizes to 15:30
PASS  '15:00' stays 15:00
PASS  'midnight' normalizes to 00:00
PASS  'Morning.' normalizes to morning
PASS  invalid time '25:00' is rejected
PASS  unknown period 'dawn' is rejected
PASS  creating a task succeeds
PASS  a task without a time stores NULL time and NULL period
PASS  an exact time is stored as HH:MM
PASS  display shows a readable 12-hour time
PASS  a period is stored separately from time
PASS  display shows the approximate period
PASS  time + period together is rejected
PASS  an empty title is rejected
PASS  date filter returns exactly that day's pending tasks
PASS  ordering: untimed tasks first, then exact times
PASS  period filter works
PASS  limit=1 returns the next task
PASS  details by partial title works
PASS  single-task completion succeeds
PASS  re-completing reports 'already completed'
PASS  a nonexistent task reports not_found
PASS  an ambiguous title return

In [13]:
print()
print("=== Error handling, confirmation safety, and retries ===")

# --- bulk completion: confirmation, cancellation, refusal during confirmation ---
bulk = fresh_service()
bulk.create_task("pay the bills", TOMORROW)
bulk.create_task("buy groceries", TOMORROW, due_time="09:00")
bulk_assistant = Assistant(service=bulk, llm=MockLLMClient())

reply = bulk_assistant.process_message("Mark all tomorrow's tasks as done")
check("bulk completion asks for confirmation first",
      "Should I mark all 2 as completed? (yes/no)" in reply
      and bulk.repo.count_tasks("pending") == 2)

reply = bulk_assistant.process_message("no")
check("cancellation leaves every task unchanged",
      "cancelled" in reply and bulk.repo.count_tasks("pending") == 2)

bulk_assistant.process_message("Mark all tomorrow's tasks as done")
reply = bulk_assistant.process_message("yes")
check("explicit 'yes' completes exactly the confirmed tasks",
      reply.startswith("Completed 2 task(s):") and bulk.repo.count_tasks("pending") == 0)

bulk.create_task("water plants", TOMORROW)
bulk_assistant.process_message("Mark all tomorrow's tasks as done")
reply = bulk_assistant.process_message("delete all tasks")
check("a destructive reply during confirmation is refused and changes nothing",
      "can't delete" in reply and bulk.repo.count_tasks("pending") == 1)

# --- bulk deletion refusal at assistant level (before any model call) ---
guard_service = fresh_service()
guard_service.create_task("keep me", TOMORROW)
guard_assistant = Assistant(service=guard_service, llm=MockLLMClient())
reply = guard_assistant.process_message("clear all tasks")
check("bulk deletion is refused explicitly",
      "can't delete tasks" in reply and guard_service.repo.count_tasks("pending") == 1)

# --- retry policy with stubbed errors (no network) ---
attempts = {"n": 0}


def flaky_call():
    attempts["n"] += 1
    if attempts["n"] <= 2:
        raise StubAPIError(503)
    return "ok"


delays = []
result = retry_with_backoff(flaky_call, max_retries=3, base_delay=0.5,
                            sleep=delays.append)
check("transient 503 errors are retried with exponential backoff",
      result == "ok" and attempts["n"] == 3 and delays == [0.5, 1.0])


def always_429():
    raise StubAPIError(429)


no_sleep = lambda delay: None   # noqa: E731 - avoid real waiting in tests
error = _raises(lambda: retry_with_backoff(always_429, max_retries=2,
                                           base_delay=0.1, sleep=no_sleep))
check("repeated 429 exhausts retries with a friendly message",
      isinstance(error, RetryExhaustedError)
      and "No task changes were made" in str(error))


def always_400():
    raise StubAPIError(400)


sleeps = []
error_400 = _raises(lambda: retry_with_backoff(always_400, max_retries=3,
                                               base_delay=0.1, sleep=sleeps.append))
check("HTTP 400 is never retried",
      isinstance(error_400, StubAPIError) and sleeps == [])

# --- a failed LLM request must never look like success ---
fail_service = fresh_service()
fail_service.create_task("keep me safe", TOMORROW)
fail_assistant = Assistant(service=fail_service, llm=FailingLLMClient(code=503))
reply = fail_assistant.process_message("Add a task to book a flight by tomorrow")
check("a failed LLM request reports the failure, not task success",
      "temporarily unavailable" in reply and "No task changes were made" in reply
      and fail_service.repo.count_tasks() == 1)

# --- invalid/malicious tool calls are rejected before execution ---
check("create_task without a date is rejected",
      validate_tool_arguments("create_task", {"title": "x"})[0] is False)
check("time + period in one call is rejected",
      validate_tool_arguments("create_task", {
          "title": "x", "due_date": "2026-01-01",
          "due_time": "10:00", "due_period": "morning"})[0] is False)
check("model-invented tool names are rejected",
      validate_tool_arguments("delete_all_tasks", {})[0] is False)

# --- ambiguity handling end-to-end through the assistant ---
amb_service = fresh_service()
amb_service.create_task("finish the report", TOMORROW)
time.sleep(0.01)
amb_service.create_task("email the report", TOMORROW)
amb_assistant = Assistant(service=amb_service, llm=MockLLMClient())
reply = amb_assistant.process_message("Mark the report as completed")
check("ambiguous task reference asks which one",
      "found multiple matching tasks" in reply and amb_service.repo.count_tasks("pending") == 2)
reply = amb_assistant.process_message("1")
check("numeric clarification completes exactly one task",
      reply.startswith("Task marked completed:") and amb_service.repo.count_tasks("pending") == 1)

print()
passed = len(TOTAL_CHECKS) - len(FAILED)
print(f"{passed}/{len(TOTAL_CHECKS)} notebook checks passed")
for name, detail in FAILED:
    print(f"  FAILED: {name} {detail}")
assert not FAILED, "Notebook self-checks failed"
print("All notebook checks passed. (This is the notebook's own compact suite, "
      "not the GitHub project's 312-test pytest suite.)")


=== Error handling, confirmation safety, and retries ===
PASS  bulk completion asks for confirmation first
PASS  cancellation leaves every task unchanged
PASS  explicit 'yes' completes exactly the confirmed tasks
PASS  a destructive reply during confirmation is refused and changes nothing
PASS  bulk deletion is refused explicitly
PASS  transient 503 errors are retried with exponential backoff
PASS  repeated 429 exhausts retries with a friendly message
PASS  HTTP 400 is never retried
PASS  a failed LLM request reports the failure, not task success
PASS  create_task without a date is rejected
PASS  time + period in one call is rejected
PASS  model-invented tool names are rejected
PASS  ambiguous task reference asks which one
PASS  numeric clarification completes exactly one task

41/41 notebook checks passed
All notebook checks passed. (This is the notebook's own compact suite, not the GitHub project's 312-test pytest suite.)


## 7. Limitations and possible improvements

**Known limitations (kept consistent with the GitHub project)**

- **Conversation state is in memory.** A pending clarification or bulk confirmation is lost if
  the assistant is restarted.
- **Lexical task matching.** Ambiguity is resolved by exact/substring/word matching, not by
  semantics; two tasks with identical titles require the full title to disambiguate.
- **Approximate periods stay approximate.** `evening` is stored as a word and never refined
  into a time, which is intentional but means no reminder scheduling exists.
- **One real provider.** Gemini is the only supported live provider; adding another requires a
  new adapter (message/tool mapping, error handling, configuration, tests).
- **Retry demonstration is stubbed.** The retry/backoff logic is shown with stubs; a real
  429/503 from Gemini would follow the same code path in the GitHub project.
- **No deletion, by design.** Users who need to remove data must do it outside the assistant.
- **Notebook-specific simplification.** An invalid `TIMEZONE` value falls back to UTC here,
  while the project fails fast with a clear error.

**Possible improvements**

- Persist conversation state so confirmations survive restarts.
- Better date parsing ("next Friday after lunch") and semantic/embedding-based task matching.
- Reminders and notifications for exact times.
- Multi-user support with per-user databases and authentication.
- Optional additional LLM adapters behind the same five-tool contract.

## 8. Summary

This notebook implemented and demonstrated a complete, self-contained version of the
AI-Powered Personal Task Manager:

1. **Natural-language task creation** with clean titles and structured due dates.
2. **Exact times** (`3 PM` → stored `15:00`, shown `3:00 PM`) and **approximate periods**
   (`tomorrow morning` → `morning`), never mixed.
3. **Clarification instead of guessing**: "at 3" asks AM/PM; ambiguous task references ask
   which task is meant.
4. **Validated tool calling**: the model can only request five operations, and Python validates
   every argument.
5. **Confirmed bulk completion**: candidates are listed first; only an explicit "yes" completes
   them, and "no" changes nothing.
6. **Safety**: bulk deletion is refused explicitly and cannot succeed — there is no deletion
   tool at all.
7. **Honest failure handling**: transient 429/503 errors are retried with bounded backoff, and a
   failed LLM request never reports task success.

The same behavior is implemented as a modular GitHub project (`src/`) with its own pytest
suite. The two deliverables are independent: this notebook imports nothing from the project,
and the project never executes the notebook.

In [14]:
# =========================================================================
# CLEANUP — remove the temporary directory created at the start
# =========================================================================
shutil.rmtree(WORKDIR, ignore_errors=True)

print(f"Temporary working directory removed: {not WORKDIR.exists()}")
print()
print("Summary:")
print(f"  - {passed}/{len(TOTAL_CHECKS)} notebook checks passed")
print("  - No live API calls were made (mock demonstrations + stubbed errors only).")
print("  - The GitHub project's src/, tests/, CLI, and config files were not required.")

Temporary working directory removed: True

Summary:
  - 41/41 notebook checks passed
  - No live API calls were made (mock demonstrations + stubbed errors only).
  - The GitHub project's src/, tests/, CLI, and config files were not required.
